In [1]:
!pip install sentence_transformers==5.2.2 numpy==2.0.2 scikit-learn==1.6.1 llmlingua==0.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 13.3 MB/s eta 0:00:0000:01
  Attempting uninstall: sentence_transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-5.2.3:
      Successfully uninstalled sentence-transformers-5.2.3


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "LLMs have limited context windows.",
    "This makes naive prompt stuffing impractical.",
    "So we rely on retrieval-augmented generation (RAG) to bring only relevant context.",
    "Chunking is a core retrieval strategy in RAG pipelines.",
    "Chunks are embedded into vectors that capture semantic meaning.",
    "Vector databases store embeddings efficiently and support fast lookup.",
    "At query time, approximate nearest neighbor (ANN) search retrieves candidate chunks.",
    "A reranker can reorder retrieved chunks for higher precision.",
    "Monitoring is critical once models are deployed.",
    "You track latency, quality drift, and retrieval hit-rates to catch regressions early.",
]

embeddings = model.encode(sentences)

SIM_THRESHOLD = 0.55

chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        embeddings[i-1].reshape(1, -1),
        embeddings[i].reshape(1, -1)
    )[0][0]

    if sim < SIM_THRESHOLD:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]
    else:
        current_chunk.append(sentences[i])

chunks.append(" ".join(current_chunk))

for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---\n{c}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Chunk 1 ---
LLMs have limited context windows.

--- Chunk 2 ---
This makes naive prompt stuffing impractical.

--- Chunk 3 ---
So we rely on retrieval-augmented generation (RAG) to bring only relevant context. Chunking is a core retrieval strategy in RAG pipelines.

--- Chunk 4 ---
Chunks are embedded into vectors that capture semantic meaning.

--- Chunk 5 ---
Vector databases store embeddings efficiently and support fast lookup.

--- Chunk 6 ---
At query time, approximate nearest neighbor (ANN) search retrieves candidate chunks. A reranker can reorder retrieved chunks for higher precision.

--- Chunk 7 ---
Monitoring is critical once models are deployed.

--- Chunk 8 ---
You track latency, quality drift, and retrieval hit-rates to catch regressions early.


In [6]:
import ast

code = """
class CreditScorer:
    def __init__(self, model):
        self.model = model

    def preprocess(self, data):
        return data.fillna(0)

    def score(self, data):
        data = self.preprocess(data)
        return self.model.predict(data)

def helper(x):
    return x * 2
"""

tree = ast.parse(code)

chunks = []

for node in tree.body:
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
        chunk = ast.get_source_segment(code, node)
        chunks.append(chunk)

for i, c in enumerate(chunks):
    print(f"\n--- Code Chunk {i+1} ---\n{c}")


--- Code Chunk 1 ---
class CreditScorer:
    def __init__(self, model):
        self.model = model

    def preprocess(self, data):
        return data.fillna(0)

    def score(self, data):
        data = self.preprocess(data)
        return self.model.predict(data)

--- Code Chunk 2 ---
def helper(x):
    return x * 2


In [7]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer, CrossEncoder


# Sample corpus (already chunked)
chunks = [
    # Chunk 0
    "Chunking is how we split long documents into smaller pieces so retrieval works within an LLM’s context window. "
    "A chunk should be big enough to be meaningful but small enough to fit alongside other evidence.",

    # Chunk 1
    "Fixed-size chunking (e.g., every 500 tokens) is simple but often cuts semantic units and loses coherence. "
    "Overlaps can reduce boundary loss but increase index size.",

    # Chunk 2
    "Semantic chunking uses embeddings to detect topic shifts between adjacent sentences. "
    "When similarity drops below a threshold, we create a boundary.",

    # Chunk 3
    "For codebases, text chunking is risky because splitting mid-function breaks syntax and meaning. "
    "AST-based chunking keeps functions and classes intact.",

    # Chunk 4
    "Vector search is usually a high-recall candidate generator. "
    "To improve precision, we re-rank retrieved chunks with a cross-encoder that jointly reads query and chunk.",

    # Chunk 5
    "Context compression reduces prompt size by removing low-information tokens. "
    "One approach uses a smaller LM to estimate token predictability and prune highly predictable tokens.",

    # Chunk 6 (distractor)
    "Monitoring in production includes latency, cost, quality, and drift. "
    "Feedback loops can trigger retraining or data refresh.",

    # Chunk 7 (distractor)
    "Prompt templates help standardize outputs. "
    "They can include instructions, examples, and structured formatting constraints."
]

# Query
query = "How do we chunk documents for LLMs?"

print("QUERY")
print(query)

# Bi-encoder retrieval (candidate generation)
print("STAGE 1: BI-ENCODER RETRIEVAL")

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = bi_encoder.encode(
    chunks,
    normalize_embeddings=True
)

query_embedding = bi_encoder.encode(
    [query],
    normalize_embeddings=True
)

scores = cosine_similarity(query_embedding, chunk_embeddings)[0]

print("\nCosine similarity vs each chunk:")
for i, s in enumerate(scores):
    print(f"  chunk[{i}]  score={s:.4f}  | {chunks[i][:80]}...")

# Select top-k candidates
top_k = 6
top_indices = np.argsort(scores)[-top_k:][::-1]

candidates = [
    (i, chunks[i], float(scores[i]))
    for i in top_indices
]

print(f"\nTop-{top_k} candidates:")
for rank, (idx, text, score) in enumerate(candidates, start=1):
    print(f"\n  #{rank}  chunk[{idx}]  cos={score:.4f}")
    print(f"  {text}")


# Cross-encoder re-ranking (precision)
print("STAGE 2: CROSS-ENCODER RE-RANKING")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [(query, text) for (_, text, _) in candidates]
rerank_scores = reranker.predict(pairs)

ranked = sorted(
    [
        (
            candidates[i][0],   # chunk index
            candidates[i][1],   # chunk text
            candidates[i][2],   # cosine score
            float(rerank_scores[i])  # cross-encoder score
        )
        for i in range(len(candidates))
    ],
    key=lambda x: x[3],
    reverse=True
)

print("\nRe-ranked candidates:")
for rank, (idx, text, cos_s, ce_s) in enumerate(ranked, start=1):
    print(f"\n  #{rank}  chunk[{idx}]")
    print(f"     cross-enc = {ce_s:.4f}")
    print(f"     cosine    = {cos_s:.4f}")
    print(f"     {text}")


# Final context selection
confidence_threshold = 0.25
top_n = 3

final_context = [
    text
    for (_, text, _, ce_s) in ranked
    if ce_s >= confidence_threshold
][:top_n]

print("FINAL SELECTED CONTEXT")
print(f"threshold={confidence_threshold}, top_n={top_n}")

for i, text in enumerate(final_context, start=1):
    print(f"\n  CONTEXT #{i}")
    print(f"  {text}")

QUERY
How do we chunk documents for LLMs?
STAGE 1: BI-ENCODER RETRIEVAL


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Cosine similarity vs each chunk:
  chunk[0]  score=0.7365  | Chunking is how we split long documents into smaller pieces so retrieval works w...
  chunk[1]  score=0.4834  | Fixed-size chunking (e.g., every 500 tokens) is simple but often cuts semantic u...
  chunk[2]  score=0.3889  | Semantic chunking uses embeddings to detect topic shifts between adjacent senten...
  chunk[3]  score=0.3751  | For codebases, text chunking is risky because splitting mid-function breaks synt...
  chunk[4]  score=0.3340  | Vector search is usually a high-recall candidate generator. To improve precision...
  chunk[5]  score=0.2912  | Context compression reduces prompt size by removing low-information tokens. One ...
  chunk[6]  score=0.0262  | Monitoring in production includes latency, cost, quality, and drift. Feedback lo...
  chunk[7]  score=0.2729  | Prompt templates help standardize outputs. They can include instructions, exampl...

Top-6 candidates:

  #1  chunk[0]  cos=0.7365
  Chunking is how we sp

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


Re-ranked candidates:

  #1  chunk[0]
     cross-enc = 7.5398
     cosine    = 0.7365
     Chunking is how we split long documents into smaller pieces so retrieval works within an LLM’s context window. A chunk should be big enough to be meaningful but small enough to fit alongside other evidence.

  #2  chunk[2]
     cross-enc = -4.0214
     cosine    = 0.3889
     Semantic chunking uses embeddings to detect topic shifts between adjacent sentences. When similarity drops below a threshold, we create a boundary.

  #3  chunk[3]
     cross-enc = -4.5678
     cosine    = 0.3751
     For codebases, text chunking is risky because splitting mid-function breaks syntax and meaning. AST-based chunking keeps functions and classes intact.

  #4  chunk[4]
     cross-enc = -5.1176
     cosine    = 0.3340
     Vector search is usually a high-recall candidate generator. To improve precision, we re-rank retrieved chunks with a cross-encoder that jointly reads query and chunk.

  #5  chunk[1]
     cros